# start

In [86]:
import pandas as pd
from pathlib import Path

In [87]:
# to be cleaned: games_data.csv, steamspy_raw.json, game_ids.csv, and game-level review summary
ROOT = Path.cwd().parent # works only if this file is kept in a subdirectory of root
RAW_DATA_PATH = ROOT / "data" / "raw"
RAW_GAME_IDS_FILE = RAW_DATA_PATH / "game_ids.csv"
RAW_GAMES_DATA_FILE = RAW_DATA_PATH / "games_data.csv"
RAW_STEAMSPY_FILE = RAW_DATA_PATH / "steamspy_raw.json"
RAW_STEAM_REVIEWS_FILE = RAW_DATA_PATH / "steam_reviews_summary.csv"

# games_id.csv cleaning

In [88]:
df = pd.read_csv(RAW_GAME_IDS_FILE)
df[df.isna().any(axis=1)]

Empty DataFrame
Columns: [appid, name, source_page]
Index: []

no NaN values, so this file should be good

# games_data.csv cleaning

## missing values

There is one row with a NaN value

In [89]:
df = pd.read_csv(RAW_GAMES_DATA_FILE)
df[df.isna().any(axis=1)]

       appid  type                             name  \
0     453480  game                  Shadowverse CCG   
1     466240  game                           Deceit   
7    1818750  game                      MultiVersus   
13     63380  game                  Sniper Elite V2   
37   1063730  game              New World: Aeternum   
45     17480  game  Command & Conquer™ Red Alert™ 3   
50   1811260  game               EA SPORTS™ FIFA 23   
63    681660  game                     Bless Online   
89   1894790  game     Stones Keeper: King Aurelius   
97   2558700  game             The Backrooms Deluxe   
114   432260  game                         Tiltagon   
137  1703080  game                F-15 Strike Eagle   
142   699760  game             Triple Otakus Puzzle   
147  1595770  game                         SUFFER 2   
154  1686540  game                   my fluffy life   

                                                genres  \
0                               Strategy; Free To Play   
1  

The game Sniper Elite V2 has no release date since it was delisted from Steam. So we'll remove it:

In [90]:
print("rows before:", len(df))
df = df[df['appid'] != 63380]
print("rows after:", len(df))

rows before: 200
rows after: 199


## duplicates

In [91]:
df[df.duplicated()]

Empty DataFrame
Columns: [appid, type, name, genres, categories, initial_price, final_price, is_free, release_date, description]
Index: []

there doesn't seem to be any duplicates

## invalid data types

In [92]:
df.dtypes

appid              int64
type                 str
name                 str
genres               str
categories           str
initial_price    float64
final_price      float64
is_free             bool
release_date         str
description          str
dtype: object

`release_date` should be a datetime object

In [93]:
df['release_date'] = pd.to_datetime(df['release_date'])
df.dtypes

appid                     int64
type                        str
name                        str
genres                      str
categories                  str
initial_price           float64
final_price             float64
is_free                    bool
release_date     datetime64[us]
description                 str
dtype: object

## inconsistent formatting
the `genres` and `categories` columns' datatypes were originally strings, and the list of genres was just a single string with semicolons in it. This will turn them into arrays

In [94]:
df['genres_list'] = df['genres'].str.split(r'\s*;\s*')
df['categories_list'] = df['categories'].str.split(r'\s*;\s*')
if not df['genres_list'].isna().all() and not df['categories_list'].isna().all():
    df['genres'] = df.pop('genres_list')
    df['categories'] = df.pop('categories_list')


df

       appid  type                                             name  \
0     453480  game                                  Shadowverse CCG   
1     466240  game                                           Deceit   
2     252490  game                                             Rust   
3     424370  game                          Wolcen: Lords of Mayhem   
4     356190  game                    Middle-earth™: Shadow of War™   
..       ...   ...                                              ...   
195   774531  game         Snail Trek - Chapter 4: The Final Fondue   
196  3205190  game                                  Alice's Lullaby   
197   682450  game                              Garrison: Archangel   
198   680680  game                                     All Haze Eve   
199  2658720  game  Vampire: The Masquerade - Reckoning of New York   

                                                genres  \
0                             [Strategy, Free To Play]   
1                        [Actio

## standardizing genres
I plan to create a new column called `Primary Genre` that has the main genre a game is.
- There are a couple of genres in the genre list that are not needed, such as Indie and Casual, however there are games where their only listed genre is one of either genre. I will just remove them.

In [ ]:
genres_condition = (df['genres'].str.len() == 1) & (
    (df['genres'].apply(lambda x: "Casual" in x)) | (df['genres'].apply(lambda x: "Indie" in x))
)

df[genres_condition]

       appid  type                         name    genres  \
22    470220  game                          UNO  [Casual]   
68   2198680  game              Dead Town Tales   [Indie]   
79    717950  game                Rainbow Snake  [Casual]   
86    836450  game          Heaven Will Be Mine   [Indie]   
129  1359900  game                 Gone Golfing   [Indie]   
141   701880  game                 Rocka Feller   [Indie]   
146  1382310  game                     Splinter   [Indie]   
171  1395830  game                      Hoplegs   [Indie]   
172   698030  game        Comit the Astrodian 3   [Indie]   
183   968290  game  Deported 2: Build That Wall   [Indie]   

                                            categories  initial_price  \
22   [Single-player, Multi-player, PvP, Online PvP,...           9.99   
68   [Single-player, Steam Achievements, Steam Clou...          11.99   
79   [Single-player, Steam Achievements, Family Sha...           2.99   
86   [Single-player, Steam Achieveme

In [96]:
df = df[~genres_condition]

## standardizing categories
Categories have either a "single-player" or "multi-player" or both in their array. I'll make new columns that is either true or false for these categories

In [97]:
df["singleplayer"] = df["categories"].apply(lambda x: "Single-player" in x)
df["multiplayer"] = df["categories"].apply(lambda x: "Multi-player" in x)


There's a program named Soundpad that is erroneously listed as a game, and isn't listed as either singleplayer nor multiplayer, so I'll remove it

In [98]:
df[(df['singleplayer'] == False) & (df['multiplayer'] == False)]

     appid  type      name                         genres        categories  \
30  629520  game  Soundpad  [Audio Production, Utilities]  [Family Sharing]   

    initial_price  final_price  is_free release_date  \
30           4.99         4.99    False   2017-09-30   

                                          description  singleplayer  \
30  Play sounds in voice chats in high digital qua...         False   

    multiplayer  
30        False  

In [99]:
df = df[df['appid'] != 629520]